## Running VGAE / Graph-PCA & baseline models

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation $z \in \mathbb{R}^{N \times D}$: representation, $u \in \mathbb{R}^{N \times 1}$: interpretability as "trajectory / gradient")
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [ ]:
import os
import gc
import sys
import pickle
import gzip

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import baseline, sb_vgae, model_train, model_eval

from torch_geometric.loader import DataLoader

### VGAE (Xenium-DESI integration)

- Integrate DESI prior to guide latent prob. clustering of Xenium

In [ ]:
import json
import squidpy as sq

from util import IO, utils, gen_graph
from models import baseline, dataset
from torch_geometric import utils as pyg_utils

from importlib import reload
reload(IO)
reload(utils)
reload(baseline)
reload(gen_graph)
reload(dataset)


In [ ]:
# Load paired Xenium & DESI

xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
sample_id = 'NIH_F5'

adata = IO.load_xenium(os.path.join(xenium_path, sample_id))  # Load xenium
adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)  # Load DESI
adata, adata_desi = IO.filter_cells(adata, adata_desi)  # Filter common cells


In [ ]:
# Compute aux. DESI expressions `u`
# Experiment: Encoding w/ raw, autoencoder, PCA & Archetype

def get_PCs(adata, coords, 
            n_pcs, k=8, 
            graph_regularize=False):
    """
    Dimension reduction w/ (graph-regularized) PCA
    """
    if graph_regularize:
        G = gen_graph.construct_graph(coords, k=k, weighted=False)  
        edge_index = pyg_utils.from_networkx(G).edge_index
        model = baseline.GPCALayer(c_in=adata.X.shape[-1],
                                   c_out=n_pcs,
                                   center=True,
                                   init_weight=True,
                                   ortho_weight=True)
        U_gpca = model(torch.tensor(adata.X).float(), edge_index)
        adata.obsm['X_pca'] = U_gpca.detach().cpu().numpy()
    else:
        sc.pp.pca(adata, n_pcs)
    return None

# %%pip install py_pcha
from py_pcha import PCHA
def get_archetypes(adata, n_archetypes, verbose=False):
    expr = adata.X if isinstance(adata.X, np.ndarray) else adata.X.A
    archetype, _, _, _, ev = PCHA(expr, noc=n_archetypes)
    adata.obsm['X_arche'] = archetype.A
    if verbose:
        print('{0} archetypes has EV={1}'.format(n_archetypes, ev))
    return None
    

n_aux = 10
# get_PCs(adata_desi, desi_coords, 
#         n_pcs=n_aux,
#         graph_regularize=False)

get_archetypes(adata_desi, n_archetypes=n_aux, verbose=True)

# Append aux. DESI expression (PC) to Xenium
# adata.obsm['X_aux'] = adata_desi.obsm['X_pca']
adata.obsm['X_aux'] = adata_desi.obsm['X_arche'].astype(np.float32)

gc.collect()

In [ ]:
arche_labels = ['Arch_'+str(i) for i in range(n_aux)]
for i in range(adata_desi.obsm['X_arche'].shape[-1]):
    adata_desi.obs['Arch_'+str(i)] = adata_desi.obsm['X_arche'][:, i]
sq.pl.spatial_scatter(adata_desi, color=arche_labels, img=False, size=1, cmap='RdBu_r')

#### Model sketch iVAE: 
- $z_i \mid u_i \sim \mathcal{Dir}(f_{\lambda}(u_i))$ vs. $z_i \mid u_i \sim \mathcal{vMF}(f_{\lambda}(u_i))$
- $x_i \mid z_i, \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i, \mathbf{A}, \theta_g)$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyro

from torch_geometric.data import DataLoader
from models import logit_vgae, model_train

In [ ]:
xenium_loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=10)
xenium_data = xenium_loader.load_graphs([adata])
xenium_train_dl = DataLoader(xenium_data, shuffle=True)

In [ ]:
from importlib import reload

reload(IO)
reload(utils)
reload(plot)
reload(dataset)
reload(configs)
reload(logit_vgae)
reload(model_train)

torch.manual_seed(42)

In [ ]:
lr = 1e-2
n_epochs = 200
device = torch.device('cuda')
n_aux = adata.obsm['X_aux'].shape[1]

train_configs = configs.set_train_configs(n_epochs=n_epochs, 
                                          lr=lr,
                                          gamma=0.5,
                                          annealing=False,
                                          device=device)

model_configs = configs.set_model_configs(device=device,
                                          prior='vMF',
                                          beta=0.5,
                                          c_in=adata.shape[1],
                                          c_aux=n_aux,            
                                          c_hidden=8,
                                          c_latent=8,
                                          k_hop=3,
                                          dropout=0.5) 

In [ ]:
# DEBUG: test Projected Normal / Von mises-Fisher?
pyro.clear_param_store()
model = logit_vgae.LogitVGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_logit_vgae(model, xenium_train_dl, train_configs)

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(train_configs.n_epochs), losses)
plt.xlabel('Epochs')
plt.ylabel('Train ELBO')
plt.show()

In [ ]:
# Inference on full data (CPU)

xenium_coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
G = gen_graph.construct_graph(xenium_coords, 
                              k=30, 
                              r=np.inf)
edge_index = pyg_utils.from_networkx(G).edge_index

model.device = 'cpu'
model = model.to('cpu')
pred_params = model.predict(adata.X.A, adata.obsm['X_aux'], edge_index)

    
qz = pred_params.qz_params[0].detach().cpu().numpy() if model_configs.prior == 'normal' else \
     pred_params.qz_params.detach().cpu().numpy()

pz = pred_params.pz.detach().cpu().numpy()
px = pred_params.px.detach().cpu().numpy()

del G
gc.collect()
torch.cuda.empty_cache()

In [ ]:
rand_indices = np.random.choice(np.arange(adata.shape[1]), size=300, replace=False)

plt.figure(figsize=(6, 4))
plt.scatter(adata.X.A[:, rand_indices].flatten(), px[:, rand_indices].flatten(), s=1)
plt.xlabel('X')
plt.ylabel("X_reconst")
plt.show()

del rand_indices

In [ ]:
sns.clustermap(np.corrcoef(qz.T), cmap='coolwarm')

In [ ]:
# plot.disp_spatial_latents(adata, pz, ncols=4)
# plot.disp_spatial_latents(adata, qz, ncols=4)

#### Trajectory Inference

- Try both Xenium & mapped DESI

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import scFates as scf
import igraph

from util import trajectory

In [ ]:
reload(trajectory)

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
sample_id = 'NIH_F5'

In [ ]:
# Load saved latent representations
adata_qz = sc.read_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')
qz = adata_qz.X

In [ ]:
# From current model
adata_qz = sc.AnnData(qz)
sc.pp.pca(adata_qz)
sc.pp.neighbors(adata_qz)
sc.tl.umap(adata_qz, n_components=3)

In [ ]:
# PC on raw Xenium expressions
# adata_norm = adata.copy()
# sc.pp.normalize_total(adata_norm)
# sc.pp.log1p(adata_norm)

# sc.pp.pca(adata_norm)
# sc.pl.pca(adata_norm,
#           color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
#                  'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
#                  'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
#                 ],
#           size=10, 
#           cmap='RdBu_r')


In [ ]:
 # Assign learnt latent embeddings to Xenium / DESI dataset as metadata
adata.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata.obsm['X_z'] = qz

adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_desi.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_desi.obsm['X_z'] = qz

In [ ]:
# Xenium markers
sc.pl.pca(adata,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          cmap='RdBu_r')


In [ ]:
sc.pl.umap(adata,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          projection='3d', alpha=0.5,
          cmap='RdBu_r')

In [ ]:
# DESI markers
sc.pl.pca(adata_desi,
          color=['FA 20:4;O',
                 'FA 18:1; O',
                 'FA 22:5;O',
                 '808.6024 m/z ± 30 ppm',
                 '865.50838 m/z ± 50 ppm',
                 '673.49987 m/z ± 30 ppm',
                 '631.4937 m/z ± 30 ppm',
                 '346.05909 m/z ± 40 ppm',
                 '754.5545 m/z ± 30 ppm',
                 '258.11631 m/z ± 30 ppm',
                 '734.58637 m/z ± 30 ppm',
                 '703.58429 m/z ± 30 ppm',
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_desi,
          color=['FA 20:4;O',
                 'FA 18:1; O',
                 'FA 22:5;O',
                 '808.6024 m/z ± 30 ppm',
                 '865.50838 m/z ± 50 ppm',
                 '673.49987 m/z ± 30 ppm',
                 '631.4937 m/z ± 30 ppm',
                 '346.05909 m/z ± 40 ppm',
                 '754.5545 m/z ± 30 ppm',
                 '258.11631 m/z ± 30 ppm',
                 '734.58637 m/z ± 30 ppm',
                 '703.58429 m/z ± 30 ppm',
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_desi,
          color=['FA 20:4;O',
                 'FA 18:1; O',
                 'FA 22:5;O',
                 '808.6024 m/z ± 30 ppm',
                 '865.50838 m/z ± 50 ppm',
                 '673.49987 m/z ± 30 ppm',
                 '631.4937 m/z ± 30 ppm',
                 '346.05909 m/z ± 40 ppm',
                 '754.5545 m/z ± 30 ppm',
                 '258.11631 m/z ± 30 ppm',
                 '734.58637 m/z ± 30 ppm',
                 '703.58429 m/z ± 30 ppm',
                ],
          size=20,
          projection='3d', alpha=0.5,
          cmap='RdBu_r')

In [ ]:
adata_qz.write_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

---

**Pseudotime sketch:**
- Fit an Elastic principal curve on `Z`
- Compute linear path btw principal "nodes"
- Compute k-NN distance from each cell to principal nodes --> final pseudotime assignment  

In [ ]:
n_pcurve_nodes = adata_qz.shape[-1]
scf.tl.curve(adata_qz, 
             Nodes=n_pcurve_nodes, 
             epg_extend_leaves=True,
             ndims_rep=adata_qz.shape[1])

In [ ]:
scf.pl.graph(adata_qz, basis='umap', nodes=list(np.arange(n_pcurve_nodes)))

`adata.uns['graph']['F']`: K x dim(Nodes) low-dim representation of the principal points <br>
`adata.uns['graph']['B']`: dim(Nodes) x dim(Nodes) adjacency matrix of the principal points <br>
`adata.obsm['X_R']`: soft assignment of cells to principal points

In [ ]:
def disp_hammer_proj(latent, ncol=4):
    def _cartesian_to_spherical(cartesian_coords):
        x, y, z = cartesian_coords[:, 0], cartesian_coords[:, 1], cartesian_coords[:, 2]
        r = np.sqrt(x**2 + y**2 + z**2)
        theta = np.arccos(z / r)  # polar angle
        phi = np.arctan2(y, x)    # azimuthal angle
        return theta, phi
    
    def _hammer_projection(theta, phi):
        z = np.sqrt(1 + np.cos(theta) * np.cos(phi / 2))
        x_hammer = 2 * np.sqrt(2) * np.cos(theta) * np.sin(phi / 2) / z
        y_hammer = np.sqrt(2) * np.sin(theta) / z
        return x_hammer, y_hammer    

    ndim = latent.shape[1]
    theta, phi = _cartesian_to_spherical(latent)
    x_hammer, y_hammer = _hammer_projection(theta, phi)

    ncol = ncol
    nrow = ndim // ncol
    if ndim % ncol != 0:
        nrow += 1
    
    idx = 0
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2*ncol, 3*nrow))
    for r in range(nrow):
        for c in range(ncol):
            axes[r, c].axis('off')
            if idx >= ndim:
                break            
            axes[r, c].scatter(x_hammer, y_hammer, s=1, c=latent[:, idx], cmap='magma')
            axes[r, c].set_title('q'+str(idx))
            idx += 1
    
    plt.tight_layout()
    plt.show()
    return None

In [ ]:
dist_metric = 'euclidean' if model_configs.prior == 'normal' else 'geodesic'
dists = trajectory.dist_to_pcurve(adata_qz, 
                                  dist_metric)

trajectory.compute_trajectory(adata_qz, dists)

g = sns.heatmap(np.corrcoef(dists.T), cmap='coolwarm')
ax = g.axes
ax.axis('off')
ax.set_title('Distance correlations to\n Principal Nodes')
sc.pl.umap(adata_qz, color='t', cmap='RdBu', title='Pseudotime')


# Visualize spatial distribution of pseudotime on full expr. space
adata.obs['t'] = adata_qz.obs['t'].values
adata.obs['t_discrete'] = adata_qz.obs['t_discrete'].values

sq.pl.spatial_scatter(adata, color='t', 
                      cmap='RdBu_r', size=20, img=False,
                      title='Pseudotime')

sq.pl.spatial_scatter(adata, color='t_discrete', 
                      cmap='tab20', size=20, img=False,
                      title='Pseudotime\n (discrete)')

**Sketch: cell-type dynamics along trajectory**

In [ ]:
annot_df = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
annot_df = annot_df.loc[adata.obs_names]
adata.obs['cell_type'] = annot_df.values

In [ ]:
adata.obs['cell_type'].value_counts()

In [ ]:
def cell_type_dynamics(
    adata,
    cell_type, 
    show_fig=False
):
    "Dynamics of cell-type proportion along discrete zonations"
    assert 't_discrete' in adata.obs.columns and 'cell_type' in adata.obs.columns
    assert cell_type in adata.obs['cell_type'].unique()

    zonation_states = np.unique(adata.obs['t_discrete'])
    dynamics = []
    for state in zonation_states:
        summary = adata[adata.obs['t_discrete'] == state].obs['cell_type'].value_counts()
        if cell_type in summary.index:
            target_count = summary[cell_type]
            total_count = summary.sum()
            dynamics.append(target_count / total_count)
        else:
            dynamics.append(0)

    if show_fig:
        fig, ax = plt.subplots(figsize=(4, 1))
        ax.plot(zonation_states, dynamics, '.--', c='blue')
        ax.set_title(cell_type)
        plt.show()
    
    return dynamics

In [ ]:
dynamics = []
cell_types = []
for cell_type in np.unique(adata.obs['cell_type']):
    if cell_type != 'Unknown' and cell_type != 'Other':
        cell_types.append(cell_type)
        dynamics.append(cell_type_dynamics(adata, cell_type))

ncol = 4
nrow = len(np.unique(adata.obs['cell_type'])) // ncol
zonation_states = np.unique(adata.obs['t_discrete'])

tj_direction = 'PV -> CV' # Modify depending on learnt trajectory direction (could be flipped)

if len(np.unique(adata.obs['cell_type'])) % ncol != 0:
    nrow += 1

idx = 0
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*1.5))
for r in range(nrow):
    for c in range(ncol):
        if idx >= len(dynamics):
            axes[r, c].axis('off')
            continue
        axes[r, c].plot(zonation_states, dynamics[idx], '.--', c='blue')
        axes[r, c].set_title(cell_types[idx])
        axes[r, c].set_xlabel(tj_direction)
        idx += 1


plt.tight_layout()
plt.show()

del nrow, ncol, idx, r, c, zonation_states, cell_type, cell_types

**DEGs along trajectory**

#### Held-out validation

In [ ]:
sample_id = 'NIH_F3'
adata_val = IO.load_xenium(os.path.join(xenium_path, sample_id))
adata_desi_val = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
adata_val, adata_desi_val = IO.filter_cells(adata_val, adata_desi_val)

In [ ]:
import time
t0 = time.perf_counter()
get_archetypes(adata_desi_val, n_archetypes=n_aux, verbose=True)
adata_val.obsm['X_aux'] = adata_desi_val.obsm['X_arche'].astype(np.float32)
t1 = time.perf_counter()

print('AA took {}s'.format(t1-t0))
del t0, t1

In [ ]:
xenium_coords = adata_val.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
G = gen_graph.construct_graph(xenium_coords, 
                              k=30, 
                              r=np.inf)
edge_index = pyg_utils.from_networkx(G).edge_index

model.device = 'cpu'
model = model.to('cpu')
pred_params = model.predict(adata_val.X.A, adata_val.obsm['X_aux'], edge_index)

    
qz = pred_params.qz_params[0].detach().cpu().numpy() if model_configs.prior == 'normal' else \
     pred_params.qz_params.detach().cpu().numpy()

pz = pred_params.pz.detach().cpu().numpy()
px = pred_params.px.detach().cpu().numpy()

del G
gc.collect()
torch.cuda.empty_cache()

In [ ]:
adata_qz = sc.AnnData(qz)
sc.pp.pca(adata_qz)
sc.pp.neighbors(adata_qz)
sc.tl.umap(adata_qz, n_components=3)


In [ ]:
n_pcurve_nodes = adata_qz.shape[-1]
scf.tl.curve(adata_qz, 
             Nodes=n_pcurve_nodes, 
             epg_extend_leaves=True,
             ndims_rep=adata_qz.shape[1])

In [ ]:
adata_val.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_val.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_val.obsm['X_z'] = qz

In [ ]:
scf.pl.graph(adata_qz, basis='umap', nodes=list(np.arange(n_pcurve_nodes)))

In [ ]:
dist_metric = 'euclidean' if model_configs.prior == 'normal' else 'geodesic'
dists = dist_to_pcurve(adata_qz, 
                       dist_metric)

compute_trajectory(adata_qz, dists)

g = sns.heatmap(np.corrcoef(dists.T), cmap='coolwarm')
ax = g.axes
ax.axis('off')
ax.set_title('Distance correlations to\n Principal Nodes')
sc.pl.umap(adata_qz, color='t', cmap='RdBu', title='Pseudotime')


# Visualize spatial distribution of pseudotime on full expr. space
adata_val.obs['t'] = adata_qz.obs['t'].values
adata_val.obs['t_discrete'] = adata_qz.obs['t_discrete'].values

sq.pl.spatial_scatter(adata_val, color='t', 
                      cmap='RdBu_r', size=20, img=False,
                      title='Pseudotime')

sq.pl.spatial_scatter(adata_val, color='t_discrete', 
                      cmap='tab20', size=20, img=False,
                      title='Pseudotime\n (discrete)')

In [ ]:
annot_df = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
annot_df = annot_df.loc[adata_val.obs_names]
adata_val.obs['cell_type'] = annot_df.values


In [ ]:
dynamics = []
cell_types = []
for cell_type in np.unique(adata_val.obs['cell_type']):
    if cell_type != 'Unknown' and cell_type != 'Other':
        cell_types.append(cell_type)
        dynamics.append(cell_type_dynamics(adata_val, cell_type))

ncol = 4
nrow = len(np.unique(adata_val.obs['cell_type'])) // ncol
zonation_states = np.unique(adata_val.obs['t_discrete'])

tj_direction = 'PV -> CV' # Modify depending on learnt trajectory direction (could be flipped)

if len(np.unique(adata_val.obs['cell_type'])) % ncol != 0:
    nrow += 1

idx = 0
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*1.5))
for r in range(nrow):
    for c in range(ncol):
        if idx >= len(dynamics):
            axes[r, c].axis('off')
            continue
        axes[r, c].plot(zonation_states, dynamics[idx], '.--', c='blue')
        axes[r, c].set_title(cell_types[idx])
        axes[r, c].set_xlabel(tj_direction)
        idx += 1


plt.tight_layout()
plt.show()

del nrow, ncol, idx, r, c, zonation_states, cell_type, cell_types

**TI w/ scFates:**

In [ ]:
import scFates as scf

In [ ]:
# adata_qz = sc.read_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

# adata_norm.obsm['X_z'] = adata_qz.obs.values  # Sorted qz
# adata_norm.obsm['X_pca'] = adata_qz.obsm['X_pca']

# adata_desi.obsm['X_z'] = adata_qz.obs.values
# adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']

In [ ]:
adata_norm.obsm['X_z'] = qz_sorted
scf.tl.curve(adata_norm, Nodes=30, use_rep='X_z', ndims_rep=adata_norm.obsm['X_z'].shape[-1])

In [ ]:
# TI w/ root assignment
scf.tl.root(adata_norm, "MYH11")

In [ ]:
scf.tl.pseudotime(adata_norm, n_jobs=16, n_map=100, seed=0)

In [ ]:
sc.pl.pca(adata_norm, color="t", cmap="coolwarm", title="Pseudotime\n (principal curve)")

In [ ]:
# Spatial distribution of trajectories
sq.pl.spatial_scatter(adata_norm, color='t', size=20, img=False, cmap='coolwarm', title='Spatial Trajectory\n (principal curve)')

DEGs along trajectory:

In [ ]:
# Xenium
scf.tl.test_association(adata_norm, n_jobs=16)
scf.tl.test_association(adata_norm, reapply_filters=True,A_cut=.5)
scf.pl.test_association(adata_norm)
ti_sig_features = adata_norm.var[adata_norm.var.signi].index

In [ ]:
scf.tl.fit(adata_norm, n_jobs=16)

Baselines: EM clustering:

In [ ]:
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=10, random_state=0).fit(adata_norm.X.A)
gm_cov = gm.covariances_
gm_soft_assign = gm.predict_proba(adata_norm.X.A)

plot.disp_spatial_latents(adata, gm_soft_assign, ncols=2)

---